# Spectral Systems LLC - Local Offline Attestation & Formal Verifier
**Founder & Principal Architect:** Eric C. Murphy | **CAGE:** `268E1` | **SAM.gov UEI:** `VTP2JPCJU3D9` | **DOI:** `10.5281/zenodo.24047902`

100% self-contained local offline verification notebook (zero external cloud dependencies). Verifies SHA-256 provenance receipts, all 5 placed 16nm `LUT6_2` SRAM truth tables (`64/64` states), `GF(2^8)` 8-step shadow-bank Bezout-Horner deflation (`thm35`), and pure `Init.Prelude` Lean 4 proofs (`thm00..thm37`). All Patents Pending (DFARS 252.227-7018 Private-Expense IP).

In [ ]:
import os, subprocess
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
def gf256_mul(a, b):
    p = 0
    for _ in range(8):
        if b & 1: p ^= a
        hi = a & 0x80; a = (a << 1) & 0xFF
        if hi: a ^= 0x1B
        b >>= 1
    return p
roots = [0x1D, 0x2B, 0x4F, 0x83]
sig = [1] + [0]*8
for i, r in enumerate(roots):
    for j in range(i+1, 0, -1): sig[j] ^= gf256_mul(sig[j-1], r)
for r in roots:
    q = [1] + [0]*8
    for k in range(1, 9): q[k] = sig[k] ^ gf256_mul(r, q[k-1])
    assert q[8] == 0
print('[OK] Offline GF(2^8) 8-step Bezout-Horner deflation verified across active roots.')

In [ ]:
env = dict(os.environ)
elan_bin = os.path.join(os.path.expanduser('~'), '.elan', 'bin')
env['PATH'] = elan_bin + os.pathsep + env.get('PATH', '')
for lf in ('CxlSubflitSanitizerFormal.lean', 'CxlSubflitSanitizerExtensions.lean', 'CxlSubflitSanitizerV110Expansion.lean'):
    subprocess.run(['lean', os.path.join(REPO_ROOT, 'formal_lean4', lf)], env=env, check=True)
subprocess.run(['python', os.path.join(REPO_ROOT, 'tests', 'run_5pillar_repo_auditor.py')], cwd=REPO_ROOT, env=env, check=True)
print('[OK] Local offline Lean 4 (thm00..thm37) & 5-Pillar Auditor passed.')